**Multi‑Agent System using LangGraph + Groq + Tavily**

**1. Install dependencies**

In [20]:
!pip install langgraph langchain langchain-groq tavily-python gradio

**2. imports dependencies**

In [21]:
import os
import asyncio
from typing import Annotated, Sequence, List, Dict, Any, Literal, Optional, TypedDict
from dataclasses import dataclass
import gradio as gr

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from tavily import TavilyClient

3. Configuration

In [22]:

# 1. Configuration (use Colab secrets or environment variables)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or input("Groq API Key: ")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY") or input("Tavily API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

# 2. Shared LLM + Tools

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)

@tool
def web_search(query: str) -> str:
    """Search the web for current information using Tavily."""
    client = TavilyClient(api_key=TAVILY_API_KEY)
    try:
        results = client.search(query, max_results=4)
        # Format for readability
        formatted = []
        for r in results.get("results", []):
            formatted.append(f"Title: {r['title']}\nContent: {r['content']}\nURL: {r['url']}\n")
        return "\n".join(formatted) if formatted else "No results found."
    except Exception as e:
        return f"Search error: {str(e)}"

tools = [web_search]
llm_with_tools = llm.bind_tools(tools)

4. State Definition

In [23]:
# State Definition (shared across all agents)

class MultiAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    next_agent: Optional[str]           # which agent to call next
    plan: Optional[str]                 # planner's output
    research_data: Optional[str]        # raw research
    final_answer: Optional[str]         # final output

5. Helper: truncate history to avoid token limit

In [24]:
# Helper: truncate history to avoid token limit
def truncate_messages(messages: Sequence[BaseMessage], max_messages: int = 20) -> List[BaseMessage]:
    """Keep system message (if any) + last N messages."""
    truncated = []
    if messages and messages[0].type == "system":
        truncated.append(messages[0])
        start = 1
    else:
        start = 0
    truncated.extend(messages[start:][-max_messages:])
    return truncated

6. Specialized Agent Nodes

In [25]:
# Specialized Agent Nodes

def planner_node(state: MultiAgentState) -> Dict[str, Any]:
    """Breaks user request into a structured plan."""
    messages = truncate_messages(state["messages"])
    system_prompt = SystemMessage(content="""You are a planner. Given the user's request, create a step-by-step plan.
Output a clear numbered list of actions. Focus on what information is needed and the order to get it.
Do not answer the user directly, only provide the plan.""")
    response = llm.invoke([system_prompt] + messages)
    plan = response.content
    return {"plan": plan, "next_agent": "researcher", "messages": [response]}

def researcher_node(state: MultiAgentState) -> Dict[str, Any]:
    """Uses web search to gather information based on the plan."""
    plan = state.get("plan", "")
    if not plan:
        plan = state["messages"][-1].content if state["messages"] else ""

    # Ask LLM to extract search queries from the plan
    query_extraction = llm.invoke([
        SystemMessage(content="Extract 1-3 specific search queries from the plan below. Return only the queries, one per line."),
        HumanMessage(content=plan)
    ])
    queries = [q.strip() for q in query_extraction.content.split("\n") if q.strip()]

    # Execute searches
    all_results = []
    for q in queries[:2]:  # limit to 2 queries for speed
        result = web_search.invoke({"query": q})
        all_results.append(f"Query: {q}\n{result}")

    research_data = "\n\n---\n\n".join(all_results)
    return {"research_data": research_data, "next_agent": "writer", "messages": []}

def writer_node(state: MultiAgentState) -> Dict[str, Any]:
    """Writes the final answer using plan + research."""
    plan = state.get("plan", "")
    research = state.get("research_data", "")
    messages = truncate_messages(state["messages"])

    system_prompt = SystemMessage(content=f"""You are a writer. Use the following plan and research to answer the user's question.
Plan:
{plan}

Research:
{research}

Write a thorough, helpful, and accurate answer. Cite sources when possible.""")
    response = llm.invoke([system_prompt] + messages)
    return {"final_answer": response.content, "next_agent": END, "messages": [response]}

def critic_node(state: MultiAgentState) -> Dict[str, Any]:
    """Optional: reviews the final answer and suggests improvements."""
    final = state.get("final_answer", "")
    if not final:
        return {"next_agent": END}
    critique_prompt = SystemMessage(content=f"""You are a critic. Review the answer below for accuracy, completeness, and clarity.
If it needs improvement, output a revised version. If it's fine, output 'APPROVED'.
Answer:
{final}""")
    critique = llm.invoke([critique_prompt])
    if "APPROVED" not in critique.content:
        # Replace with improved version
        return {"final_answer": critique.content, "next_agent": END, "messages": [AIMessage(content=critique.content)]}
    return {"next_agent": END}


7. Router: decides which node to call next based on state

In [26]:
# Router: decides which node to call next based on state
def router(state: MultiAgentState) -> Literal["planner", "researcher", "writer", "critic", END]:
    next_agent = state.get("next_agent")
    if next_agent == "planner":
        return "planner"
    elif next_agent == "researcher":
        return "researcher"
    elif next_agent == "writer":
        return "writer"
    elif next_agent == "critic":
        return "critic"
    else:
        return END

8. Build the LangGraph workflow

In [27]:
# Build the LangGraph workflow

workflow = StateGraph(state_schema=MultiAgentState)

# Add nodes
workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.add_node("critic", critic_node)   # optional, you can omit from edges

# Entry point: planner
workflow.add_edge(START, "planner")

# Conditional routing
workflow.add_conditional_edges("planner", router)
workflow.add_conditional_edges("researcher", router)
workflow.add_conditional_edges("writer", router)
workflow.add_conditional_edges("critic", router)

# Memory (per conversation thread)
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

9. Gradio UI Wrapper (fixed streaming)

In [28]:
# Gradio UI Wrapper (fixed streaming)

async def respond(message: str, history: list):
    """
    Async generator for Gradio ChatInterface.
    Shows plan/research progress via toast, then streams final answer.
    """
    thread_id = "user_1"   # Use a session ID per user in production
    config = {"configurable": {"thread_id": thread_id}}
    inputs = {"messages": [HumanMessage(content=message)]}

    final_answer = ""
    async for event in app.astream(inputs, config, stream_mode="values"):
        # Show progress without polluting the chat
        if "plan" in event and event["plan"] and not final_answer:
            gr.Info(f"📋 Planning: {event['plan'][:120]}...")
        if "research_data" in event and event["research_data"] and not final_answer:
            gr.Info("🔍 Research completed. Writing answer...")
        # Capture final answer (could be updated by critic later)
        if "final_answer" in event and event["final_answer"]:
            final_answer = event["final_answer"]

    if not final_answer:
        yield "Sorry, I couldn't generate an answer."
    else:
        # Stream character by character for typing effect
        buffer = ""
        for ch in final_answer:
            buffer += ch
            yield buffer
            await asyncio.sleep(0.01)   # smooth typing speed

# -------------------------------------------------------------------
# 9. Launch Gradio Interface (simplified)
# -------------------------------------------------------------------
if __name__ == "__main__":
    with gr.Blocks(title="Multi-Agent Assistant", theme=gr.themes.Soft()) as demo:
        gr.Markdown("""# Multi‑Agent System
        **Planner** → **Researcher** (web search) → **Writer** → (optional Critic)
        Built with LangGraph + Groq + Tavily. Ask anything that needs up‑to‑date research!
        """)
        chatbot = gr.ChatInterface(
            fn=respond,
            title="Multi‑Agent Assistant",
            description="Type your question. The system will plan, search the web, and write a final answer.",
            examples=[
                "What are the latest developments in quantum computing?",
                "Compare LangGraph and AutoGen for multi‑agent systems.",
                "List top 3 AI conferences in 2026."
            ]
        )
    demo.launch(share=True)

/tmp/ipykernel_10330/2144008365.py:37: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Multi-Agent Assistant", theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c6ce54b4c611332daf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
